# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library and the Croissant schema. All dataset entities are referenced by their `@id` as per best practices.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

**Sections:**
1. Data Loading
2. Data Overview
3. Data Extraction
4. Exploratory Data Analysis (EDA)
5. Visualization
6. Conclusion

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata. Entities are referenced using their `@id`.

In [ ]:
# List all available record sets with their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found: {len(record_sets)}")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")
    # List fields for this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use only `@id`s.

In [ ]:
# Prepare a list of record set @id's for loading data
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Select the main record set @id for demonstration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping data. All field references use the field `@id`.

In [ ]:
# Find a numeric field for analysis from the field metadata
rs = next((rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id), None)
numeric_field = None
for field in rs.fields:
    if field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_field = field.id
        print(f"Numeric field selected: {numeric_field} ({field.name})")
        break
if not numeric_field:
    print("No numeric fields found. EDA operations may be limited.")

df = dataframes[main_record_set_id]
filtered_df = df

# For demonstration, apply a threshold filter if a numeric field exists
if numeric_field and numeric_field in df.columns:
    # Choose a sample threshold
    sample_threshold = 10
    filtered_df = df[df[numeric_field] > sample_threshold]
    print(f"Filtered records with {numeric_field} > {sample_threshold}:")
    display(filtered_df.head())
    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())
else:
    print("Skipping numeric filtering and normalization as there is no numeric field.")

# Try grouping by a non-numeric field if available
group_field = None
if rs:
    for field in rs.fields:
        if field.data_type in ['schema:Text', 'schema:Boolean'] and field.id != numeric_field:
            group_field = field.id
            print(f"Grouping by field: {group_field} ({field.name})")
            break
if group_field and group_field in filtered_df.columns and numeric_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric attribute or explore relationships between fields. All columns referenced using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of numeric field: {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
else:
    print("No numeric field available for histogram.")

# Scatter plot with the first two numeric fields if available
numeric_fields = [field.id for field in rs.fields if field.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']]
if len(numeric_fields) >= 2 and all(fld in df.columns for fld in numeric_fields[:2]):
    plt.figure(figsize=(6, 6))
    sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
    plt.title(f"Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel(numeric_fields[1])
    plt.show()

## 6. Conclusion
This notebook provided a walkthrough of loading and exploring the FAIR^2 clinical CRC dataset using the `mlcroissant` library.

**Summary:**
- The dataset's structure (record sets and fields) was explored and documented using only `@id` references.
- The main tabular data was loaded into a DataFrame for automated analysis.
- Exploratory steps such as numeric filtering, normalization, grouping, and basic visualizations were demonstrated.
- Further investigation can continue by referencing additional record sets or metadata via their Croissant `@id`s for more complex analysis.

For further information, see the [FAIR^2 data package](https://sen.science/doi/10.71728/senscience.qs2f-h81p).